In [23]:
import gymnasium as gym
import numpy as np
env = gym.make("CliffWalking-v1")
n_states = env.observation_space.n
n_actions = env.action_space.n
print("States:", n_states, "Actions:", n_actions)

States: 48 Actions: 4


In [24]:
def value_iteration(env, gamma=0.9, theta=1e-6, P=None, max_iterations=10000):
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    # P[s][a] = [(prob, next_state, reward, done), ...] là mô hình môi trường.
    # Cho phép truyền P tùy biến để thử đổi reward ở phần thảo luận.
    P = env.unwrapped.P if P is None else P
    V = np.zeros(n_states)

    def q_values(s):
        q = np.zeros(n_actions)
        for a in range(n_actions):
            for prob, s_next, r, done in P[s][a]:
                # done=True: episode kết thúc nên không còn return tương lai
                q[a] += prob * (r + gamma * V[s_next] * (not done))
        return q

    # TODO 1: vòng lặp cập nhật V(s)
    for i in range(max_iterations):
        delta = 0.0
        for s in range(n_states):
            v_old = V[s]
            V[s] = np.max(q_values(s))          # Bellman optimality update
            delta = max(delta, abs(v_old - V[s]))
        # TODO 3: tiêu chí dừng.
        # delta là thay đổi lớn nhất của V trong 1 sweep. Toán tử Bellman là phép co
        # hệ số gamma, nên khi delta < theta thì V cách V* nhiều nhất cỡ
        # theta*gamma/(1-gamma): đủ gần để policy greedy không còn đổi.
        if delta < theta:
            break
    else:
        print(f"Warning: Value iteration did not converge within {max_iterations} iterations. Delta was {delta:.2e}")

    # TODO 2: trích policy greedy từ V đã hội tụ
    policy = np.array([np.argmax(q_values(s)) for s in range(n_states)])
    return V, policy

In [25]:
def show_policy(policy):
    arrows = {0: "↑", 1: "→", 2: "↓", 3: "←"}
    for r in range(4):
        row = ""
        for c in range(12):
            s = r * 12 + c
            if r == 3 and 1 <= c <= 10:
                row += "C "                      # vực
            elif s == 47:
                row += "G "                      # đích
            else:
                row += arrows[policy[s]] + " "
        print(row)


# EVALUATION
for gamma in (0.9, 0.99):
    V, policy = value_iteration(env, gamma=gamma)
    print(f"\n=== gamma = {gamma} ===")
    print("Average state value:", np.mean(V))
    show_policy(policy)

# Thử đổi phạt rơi vực -100 -> -10 (copy P rồi sửa reward)
import copy
P_new = copy.deepcopy(env.unwrapped.P)
for s in P_new:
    for a in P_new[s]:
        P_new[s][a] = [(p, sn, -10 if r == -100 else r, d) for p, sn, r, d in P_new[s][a]]

for gamma in (0.9, 0.99):
    V, policy = value_iteration(env, gamma=gamma, P=P_new)
    print(f"\n=== phạt -10, gamma = {gamma} ===")
    print("Average state value:", np.mean(V))
    show_policy(policy)


=== gamma = 0.9 ===
Average state value: -5.088569925055769
→ → → → → → → → → → → ↓ 
→ → → → → → → → → → → ↓ 
→ → → → → → → → → → → ↓ 
↑ C C C C C C C C C C G 

=== gamma = 0.99 ===
Average state value: -7.140831912127735
→ → → → → → → → → → → ↓ 
→ → → → → → → → → → → ↓ 
→ → → → → → → → → → → ↓ 
↑ C C C C C C C C C C G 

=== phạt -10, gamma = 0.9 ===
Average state value: -5.088569925055769
→ → → → → → → → → → → ↓ 
→ → → → → → → → → → → ↓ 
→ → → → → → → → → → → ↓ 
↑ C C C C C C C C C C G 

=== phạt -10, gamma = 0.99 ===
Average state value: -7.140831912127735
→ → → → → → → → → → → ↓ 
→ → → → → → → → → → → ↓ 
→ → → → → → → → → → → ↓ 
↑ C C C C C C C C C C G 
